# 03 — Elastic Weight Consolidation (EWC)

EWC (Kirkpatrick et al., 2017) is a regularisation-based approach to continual learning.
After each task it computes the **diagonal Fisher Information Matrix** over the task's training data.
The Fisher score for a parameter estimates how much the task loss changes when that parameter moves,
i.e. it measures the parameter's importance to the current task.

During the next task, an L2 regularisation term is added to the loss:

$$\mathcal{L} = \mathcal{L}_{\text{CE}} + \frac{\lambda}{2} \sum_i F_i (\theta_i - \theta^*_i)^2$$

where $F_i$ is the accumulated Fisher score and $\theta^*_i$ is the parameter value after the previous task.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.train import run_continual
from src.evaluate import backward_transfer, forward_transfer, average_accuracy, print_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Run EWC

In [ ]:
R_ewc = run_continual(
    method='ewc',
    dataset='cifar100',
    n_tasks=20,
    epochs_per_task=10,
    lr=0.1,
    batch_size=128,
    device=device,
    ewc_lambda=5000.0,
    seed=42,
    save_dir='../results/baseline',
    verbose=True,
)

## 2. Accuracy Matrix

In [ ]:
T = R_ewc.shape[0]
fig, ax = plt.subplots(figsize=(11, 9))
display = np.where(np.isnan(R_ewc), 0.0, R_ewc * 100)
im = ax.imshow(display, vmin=0, vmax=100, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Accuracy (%)')
ax.set_xticks(range(T)); ax.set_yticks(range(T))
ax.set_xticklabels([f'T{j}' for j in range(T)], fontsize=8)
ax.set_yticklabels([f'T{i}' for i in range(T)], fontsize=8)
ax.set_xlabel('Task evaluated'); ax.set_ylabel('After training task')
ax.set_title('EWC — Accuracy Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/baseline/acc_matrix_ewc.png', bbox_inches='tight')
plt.show()

## 3. Sensitivity to λ (EWC regularisation coefficient)

In [ ]:
# Quick sweep over lambda using only 5 tasks to save time
lambdas = [100, 1000, 5000, 20000, 100000]
avg_accs, bwts = [], []

for lam in lambdas:
    print(f'  lambda={lam} ...', end='')
    R = run_continual(
        method='ewc', dataset='cifar100', n_tasks=5,
        epochs_per_task=5, lr=0.1, batch_size=128,
        device=device, ewc_lambda=lam, seed=42,
        save_dir='../results/baseline', verbose=False,
    )
    avg_accs.append(average_accuracy(R) * 100)
    bwts.append(backward_transfer(R) * 100)
    print(f' avg_acc={avg_accs[-1]:.1f}%  bwt={bwts[-1]:.1f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.semilogx(lambdas, avg_accs, marker='o', color='#f39c12', linewidth=2)
ax1.set_xlabel('λ (log scale)'); ax1.set_ylabel('Avg Accuracy (%)')
ax1.set_title('EWC: Avg Accuracy vs λ'); ax1.grid(True, alpha=0.3)

ax2.semilogx(lambdas, bwts, marker='s', color='#e74c3c', linewidth=2)
ax2.set_xlabel('λ (log scale)'); ax2.set_ylabel('BWT (%)')
ax2.set_title('EWC: Backward Transfer vs λ'); ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')

plt.suptitle('EWC Regularisation Coefficient Sweep (5 tasks)', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/baseline/ewc_lambda_sweep.png', bbox_inches='tight')
plt.show()

## 4. Metrics

In [ ]:
print_metrics(R_ewc, method='EWC (λ=5000)')

## Observations

EWC significantly reduces forgetting compared to naive fine-tuning, but the protection degrades
over many tasks because the regularisation anchors all parameters to all past optima simultaneously.
In the 20-task setting this causes interference between tasks' Fisher matrices.
Pruning-based methods (PackNet, CausalPruner) avoid this by creating disjoint sub-networks.